# Bone-Loss Segmentation - Training Notebook

**Affiliation:** [InReDD](https://inredd.com.br/en)

This notebook documents the training pipeline used in the companion paper for periodontal
bone-loss assessment from intraoral photographs. The pipeline has three stages:

1. **Mouth detection** (Detectron2 RetinaNet) used as a preprocessing crop.
2. **Crown and alveolar-ridge segmentation** - the core contribution. Two architectures are trained and compared:
   - SegFormer-B5 fine-tuned with PyTorch Lightning.
   - YOLOv11x-seg fine-tuned with Ultralytics.
3. **Result archiving.**

## Data and Model Availability

> The pretrained model weights and trained checkpoints referenced in this notebook are property of **InReDD** (https://inredd.com.br/en) and are **not publicly available**, nor distributed with this notebook. The clinical dataset is also property of InReDD; it may be requested for non-commercial academic use through the [InReDD Open Data portal](https://inredd.com.br/en/solutions/open-data). This notebook is released for **methodological transparency and reproducibility only**.

## Reproducibility

- Trained on a local workstation with an NVIDIA RTX 5070 12 GB GPU.
- Key dependencies (pinned in the install cell): `pytorch-lightning==2.4.0`, `transformers==4.46.2`, plus Detectron2 built from source and Ultralytics for YOLOv11.
- Random seeds, image size, batch size, and other hyperparameters are surfaced as named constants at the top of each section.

## Repository layout (expected for re-running locally)

```
boneloss_inference/
|-- data/
|   |-- inredd_crown_alveolar_ridge_dataset/   # raw LabelMe-style JSON + images per patient (InReDD only)
|   |-- yolo/
|       |-- crown/data.yaml
|       |-- alveolar/data.yaml
|-- models/
|   |-- inredd_mouth_detector.pth              # InReDD only
|   |-- inredd_tooth_segmenter.pth             # InReDD only
|-- notebooks/
    |-- Bone_Loss_Segmentation_(Training).ipynb
```


# Installation Sections

In [ ]:
# NOTE: InReDD property - not publicly available, not distributed with this notebook.
print("Installing Python libraries...")
!pip install -q gdown pytorch-lightning==2.4.0
!pip install -q transformers==4.46.2 datasets==2.21.0
print("Libraries installed.")

print("\nDownloading datasets and models from Google Drive...")

# --- Dataset Bone Loss ---
# Use -O for a fixed filename and -d to extract into a specific folder
DATASET_ZIP_PATH = '/content/inredd_crown_alveolar_ridge_dataset.zip'
DATASET_EXTRACT_PATH = '/content/Datasets/'
!gdown --id '<INREDD_DATASET_DRIVE_ID>'
!mkdir -p {DATASET_EXTRACT_PATH}
!unzip -q -o {DATASET_ZIP_PATH} -d {DATASET_EXTRACT_PATH}
print(f"Dataset 'Bone Loss' downloaded and extracted to: {DATASET_EXTRACT_PATH}")

# --- Model Mouth Detection ---
!gdown --id '<INREDD_MOUTH_DETECTOR_DRIVE_ID>' --quiet
!gdown --id '<INREDD_TOOTH_SEGMENTER_DRIVE_ID>' --quiet
print("Mouth-detection models downloaded.")

print("\nOrganizing model folder structure...")
!mkdir -p Models/MouthDetection
!mv inredd_mouth_detector.* Models/MouthDetection/
print("Models moved to folder 'Models/MouthDetection/'.")

print("\nCompiling and installing Detectron2 (may take a few minutes)...")

!git clone https://github.com/facebookresearch/detectron2.git > /dev/null 2>&1

# Install Detectron2 from the cloned folder, compiling C++/CUDA extensions
!pip install -e detectron2

import os

try:
    import detectron2
    print("\nDetectron2 imported successfully!")
    print("Setup complete! The session will now restart to apply the changes...")

    os.kill(os.getpid(), 9)
except ImportError:
    print("\nFailed to import Detectron2. Installation may have failed.")

# Library Imports

In [ ]:
import os
import cv2
import json
import torch
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
import pytorch_lightning as pl

from torch import nn
from tqdm import tqdm
from PIL import Image, ImageDraw
from datasets import load_metric
from detectron2 import model_zoo
from matplotlib import pyplot as plt
from detectron2.config import get_cfg
from detectron2.structures import Boxes
from detectron2.engine import DefaultPredictor
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from detectron2.export.torchscript import scripting_with_instances, freeze_training_mode

# Datasets Imports

In [ ]:
# Path to the zipped InReDD dataset (shared via Drive link)
zip_path = '/content/inredd_crown_alveolar_ridge_dataset.zip'

# Local directory to extract the files
extract_path = '/content/Datasets'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Files extracted to: {extract_path}')

# Mouth Detection Inference (Preprocessing Annotation Step)

In [ ]:
# NOTE: InReDD property - not publicly available, not distributed with this notebook.
class MouthDetector:
    """Detect a mouth bounding box in intraoral photographs using Detectron2 RetinaNet.

    Wraps a Detectron2 ``DefaultPredictor`` built from a base RetinaNet config
    and trained weights provided as a ``.pth`` file. Used as a preprocessing
    crop step before tooth segmentation.
    """
    def __init__(self, weight_path: str):
        """Build the underlying ``DefaultPredictor`` from the trained RetinaNet weights.

        Parameters
        ----------
        weight_path : str
            Path to the trained RetinaNet weights (``.pth``).
        """
        print("--- Initializing Mouth Detector (Detectron2 mode) ---")

        # 1. Configure the model
        cfg = get_cfg()
        # Use a Detectron2 base config.
        cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_101_FPN_3x.yaml"))
        cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        # 2. Load the weights
        cfg.MODEL.WEIGHTS = weight_path

        # 3. Adjust parameters specific to the trained model
        cfg.MODEL.RETINANET.NUM_CLASSES = 3  # Number of classes the model was trained on
        cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.5 # Adjust the threshold as needed

        print(f"Selected inference device: {cfg.MODEL.DEVICE}")
        print(f"Loading weights from: {weight_path}")

        # 4. Create the "predictor" object
        try:
            self.predictor = DefaultPredictor(cfg)
            print("Detectron2 model loaded successfully!")
        except Exception as e:
            print(f"ERROR creating DefaultPredictor: {e}")
            self.predictor = None

    def predict(self, image_rgb: np.ndarray):
        """Detect the mouth in an RGB image and return the highest-scoring box.

        Parameters
        ----------
        image_rgb : np.ndarray
            Image in NumPy ``(H, W, C)`` layout with RGB channel order.

        Returns
        -------
        list of float or None
            Bounding box ``[xmin, ymin, xmax, ymax]`` for the highest-scoring
            detection, or ``None`` if no mouth is detected or the predictor is
            unavailable.
        """
        if self.predictor is None:
            print("ERROR: predictor was not loaded; cannot run prediction.")
            return None

        try:
            # The Detectron2 predictor expects a BGR image, so convert it back
            image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

            # Run prediction
            with torch.no_grad():
                outputs = self.predictor(image_bgr)

            # Extract the results
            instances = outputs["instances"].to("cpu")
            pred_boxes = instances.pred_boxes.tensor.numpy()
            scores = instances.scores.numpy()

            if len(scores) > 0:
                # Find the index of the highest score (best detection)
                max_score_idx = np.argmax(scores)
                best_box = pred_boxes[max_score_idx].tolist()
                best_score = scores[max_score_idx]

                return best_box
            else:
                return None

        except Exception as e:
            print(f"ERROR during Detectron2 inference: {e}")
            return None

class Segmentacao_De_Dentes:
    def __init__(self):
        self.predictor = None
        print("[Segmentacao_De_Dentes5] Initializing...")
        self.cfg = None
        self.setup()

    def setup(self):
        modelo = "Misc/scratch_mask_rcnn_R_50_FPN_9x_gn.yaml"

        # Model configuration
        cfg = get_cfg()
        cfg.MODEL.DEVICE = "cpu"
        cfg.merge_from_file(model_zoo.get_config_file(modelo))
        cfg.MODEL.WEIGHTS = os.path.relpath("resources/models/base/inredd_tooth_segmenter.pth")
        cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.8

        self.cfg = cfg
        self.predictor = DefaultPredictor(cfg)

    def export_torchscript(self, export_path="resources/models/TorchScript/model_segmentacao_de_dentes5_torchscript.pt"):
        # Access the underlying model
        model = self.predictor.model
        model.eval()

        # Define a separate forward function without post-processing, with type annotations:
        def forward_no_postprocess(self: GeneralizedRCNN, batched_inputs: List[Dict[str, torch.Tensor]]) -> List[Instances]:
            """Forward pass without Detectron2 post-processing.

            Mirrors ``GeneralizedRCNN.forward`` but skips the post-processing
            step so the model can be exported to TorchScript with raw outputs.
            """
            images = self.preprocess_image(batched_inputs)
            features = self.backbone(images.tensor)
            # If proposal_generator needs the third argument, ensure the type is compatible.
            # If not needed, remove the `None`.
            proposals, _ = self.proposal_generator(images, features, None)
            results, _ = self.roi_heads(images, features, proposals)
            return results

        # Bind the new forward to the model
        model.forward = types.MethodType(forward_no_postprocess, model)

        self.model = model

        # Define the fields required for scripting
        fields = {
            "pred_boxes": Boxes,
            "scores": torch.Tensor,
            "pred_classes": torch.Tensor,
            "pred_masks": torch.Tensor,
            "proposal_boxes": Boxes,
            "objectness_logits": torch.Tensor
        }

        # Export the model to TorchScript using the freeze_training_mode context
        with freeze_training_mode(model):
            scripted_model = scripting_with_instances(model, fields)

        # Save the exported model
        os.makedirs(os.path.dirname(export_path), exist_ok=True)
        scripted_model.save(export_path)
        print(f"TorchScript model exported to: {export_path}")

# Export the model
detector = Segmentacao_De_Dentes()
detector.export_torchscript()

# Segformer section

In [ ]:
# NOTE: InReDD property - not publicly available, not distributed with this notebook.
# --- 1. MAIN CONFIGURATION ---

# Folder containing the original dataset
dataset_name = "inredd_crown_alveolar_ridge_dataset"
ROOT_DIR = f"/content/Datasets/{dataset_name}"

# Ratios for the dataset split
SPLIT_RATIOS = {"train": 0.7, "valid": 0.2, "test": 0.1}

# Set whether to resize images and masks.
ENABLE_RESIZE = True  # Set to False to disable
# Set the target size (WIDTH, HEIGHT) if resizing is enabled.
TARGET_SIZE = (1920, 1080)
# Enable to run detection and cropping AFTER dataset creation.
ENABLE_MOUTH_CROP = True
MOUTH_DETECTOR_WEIGHT_PATH = "/content/Models/MouthDetection/inredd_mouth_detector.pth"

# Configuration of the datasets to generate
DATASET_CONFIGS = {
    "ds_crown": ["crown"],
    "ds_alveolarridge": ["alveolar_ridge"],
    # Add as many profiles as needed
    # "dataset_mand_e_max": ["mand", "max"],
    # "dataset_coroa_e_rebordo": ["crown", "alveolar_ridge"],
    # "dataset_completo": ["mand", "max", "crown", "alveolar_ridge"]
}

# --- 2. HELPER FUNCTIONS ---
def analyze_dataset_stats(root_dir):
    """Compute image-count and average-dimension statistics for the original dataset."""
    print("--- Starting original dataset analysis ---")
    if not os.path.exists(root_dir):
        print(f"ERROR: root directory '{root_dir}' not found.")
        return [], 0, 0 # Return empty list, 0, 0

    patient_folders = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    num_patients = len(patient_folders)

    if num_patients == 0:
        print("No patient folder found.")
        return [], 0, 0 # Return empty list, 0, 0

    total_images, total_width, total_height = 0, 0, 0
    for patient in patient_folders:
        patient_path = os.path.join(root_dir, patient)
        image_files = [f for f in os.listdir(patient_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img_file in image_files:
            try:
                with Image.open(os.path.join(patient_path, img_file)) as img:
                    w, h = img.size
                    total_width += w
                    total_height += h
                    total_images += 1
            except Exception as e:
                print(f"Warning: could not read image {img_file}. Error: {e}")

    avg_images_per_patient = total_images / num_patients if num_patients > 0 else 0
    avg_width = total_width / total_images if total_images > 0 else 0
    avg_height = total_height / total_images if total_images > 0 else 0

    print(f"Total patients: {num_patients}")
    print(f"Total images: {total_images}")
    print(f"Average images per patient: {avg_images_per_patient:.2f}")
    print(f"Average image size: {avg_width:.0f} x {avg_height:.0f} (Width x Height)\n")

    return patient_folders, num_patients, total_images # Return 3 values

def create_output_structure(output_dir, splits):
    """Create the output folder layout (``train`` / ``valid`` / ``test``)."""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    for split in splits:
        os.makedirs(os.path.join(output_dir, split), exist_ok=True)

def create_class_csv(output_dir, splits, class_map):
    """Write the ``_classes.csv`` class-to-pixel-value mapping inside every split folder."""
    class_data = [{"Class": name, "Pixel Value": value} for name, value in class_map.items()]
    class_data.insert(0, {"Class": "background", "Pixel Value": 0})
    df = pd.DataFrame(class_data)
    for split in splits:
        df.to_csv(os.path.join(output_dir, split, "_classes.csv"), index=False)

def process_and_convert_files(patient_list, source_root_dir, dest_split_dir, class_map, resize_enabled, target_size):
    """Read LabelMe JSON, optionally resize, build segmentation masks, and save image/mask pairs."""
    for patient_id in patient_list:
        patient_path = os.path.join(source_root_dir, patient_id)
        for json_file in [f for f in os.listdir(patient_path) if f.endswith('.json')]:
            json_path = os.path.join(patient_path, json_file)
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                source_image_path = os.path.join(patient_path, data['imagePath'])
                original_img = Image.open(source_image_path)
                original_width, original_height = original_img.size

                final_img = original_img
                final_shapes = data['shapes']
                final_width, final_height = original_width, original_height

                # RESIZE LOGIC
                if resize_enabled:
                    final_img = original_img.resize(target_size, Image.Resampling.LANCZOS)
                    final_width, final_height = target_size

                    x_scale = final_width / original_width
                    y_scale = final_height / original_height

                    # Scale the polygon coordinates
                    scaled_shapes = []
                    for shape in data['shapes']:
                        new_shape = shape.copy()
                        new_shape['points'] = [[p[0] * x_scale, p[1] * y_scale] for p in shape['points']]
                        scaled_shapes.append(new_shape)
                    final_shapes = scaled_shapes

                # Create the mask using the final dimensions and shapes
                mask = Image.new('L', (final_width, final_height), 0)
                draw = ImageDraw.Draw(mask)

                for shape in final_shapes:
                    label = shape['label']
                    if label in class_map:
                        points = [tuple(p) for p in shape['points']]
                        if len(points) > 2:
                            pixel_value = class_map[label]
                            draw.polygon(points, outline=pixel_value, fill=pixel_value)

                # Save final files
                base_name = os.path.splitext(data['imagePath'])[0]
                mask_name = f"{base_name}_mask.png"
                dest_image_path = os.path.join(dest_split_dir, data['imagePath'])
                dest_mask_path = os.path.join(dest_split_dir, mask_name)

                final_img.convert("RGB").save(dest_image_path)  # .convert("RGB") prevents alpha-channel issues (PNG)
                mask.save(dest_mask_path)

            except Exception as e:
                print(f"Error processing file {json_path}: {e}")

def print_summary_report(output_dir, total_images, splits):
    """Print the per-split image distribution for a generated dataset."""
    print("\n--- Distribution report ---")
    for split in splits:
        split_path = os.path.join(output_dir, split)
        image_files_in_split = len([f for f in os.listdir(split_path) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and '_mask' not in f])
        percentage = (image_files_in_split / total_images) * 100 if total_images > 0 else 0
        print(f"Folder '{split}':\t{image_files_in_split} images ({percentage:.2f}%)")
    print("-" * 33)

def crop_dataset_in_place(dataset_dir, detector, splits):
    """Run mouth detection on an existing split and crop both image and mask in place.

    Iterates over each split folder of an already-generated dataset, predicts
    a mouth bounding box per image, and overwrites both the image and its
    paired mask with the cropped versions.
    """
    print(f"Starting crop for dataset at: '{dataset_dir}'")
    for split in splits:
        split_path = os.path.join(dataset_dir, split)
        print(f"\nProcessing split: '{split}'")

        # List only image files (not masks)
        image_files = [f for f in os.listdir(split_path) if f.lower().endswith(('.jpg', '.png')) and '_mask' not in f]

        for image_name in tqdm(image_files, desc=f"Recortando {split}"):
            base_name = os.path.splitext(image_name)[0]
            mask_name = f"{base_name}_mask.png"

            image_path = os.path.join(split_path, image_name)
            mask_path = os.path.join(split_path, mask_name)

            if not os.path.exists(mask_path):
                print(f"Warning: mask '{mask_name}' not found for image '{image_name}'. Skipping.")
                continue

            try:
                # Load the image for inference
                image_bgr = cv2.imread(image_path)
                if image_bgr is None: continue

                # The detector expects RGB
                image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                box = detector.predict(image_rgb)

                if box:
                    # Load the mask (grayscale to preserve pixel values)
                    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                    if mask is None: continue

                    h, w, _ = image_bgr.shape
                    xmin, ymin, xmax, ymax = map(int, [max(0, box[0]), max(0, box[1]), min(w, box[2]), min(h, box[3])])

                    # Crop image and mask using the SAME coordinates
                    cropped_image = image_bgr[ymin:ymax, xmin:xmax]
                    cropped_mask = mask[ymin:ymax, xmin:xmax]

                    # Overwrite the original files with the cropped versions
                    cv2.imwrite(image_path, cropped_image)
                    cv2.imwrite(mask_path, cropped_mask)

            except Exception as e:
                print(f"Error cropping {image_name}: {e}")

if __name__ == "__main__":
    # Step 1: analysis and patient split
    patients, num_patients, total_images = analyze_dataset_stats(ROOT_DIR)

    if not patients:
        print("No patient found; script will exit.")
    else:
        print("--- Splitting patients into sets (same split applied to all datasets) ---")
        random.shuffle(patients)
        train_count = int(num_patients * SPLIT_RATIOS['train'])
        valid_count = int(num_patients * SPLIT_RATIOS['valid'])
        train_patients = patients[:train_count]
        valid_patients = patients[train_count : train_count + valid_count]
        test_patients = patients[train_count + valid_count:]
        print(f"Patients - train: {len(train_patients)}, validation: {len(valid_patients)}, test: {len(test_patients)}\n")

        # Step 2: initialize the mouth-detection model (once only)
        detector = None
        if ENABLE_MOUTH_CROP:
            detector = MouthDetector(MOUTH_DETECTOR_WEIGHT_PATH)
            if detector.predictor is None:
                print("WARNING: mouth-detection model could not be loaded; crop step will be skipped.")
                ENABLE_MOUTH_CROP = False

        # Step 3: loop to generate each configured dataset
        for config_name, desired_classes in DATASET_CONFIGS.items():
            print("="*60)
            print(f"====== GENERATING DATASET: '{config_name}' ======")
            print(f"Included classes: {desired_classes}")
            if ENABLE_RESIZE:
                print(f"Resize ENABLED to {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")
            else:
                print("Resize DISABLED")
            print("="*60)

            current_output_dir = f"/content/Datasets/{config_name}"
            current_class_map = {class_name: i + 1 for i, class_name in enumerate(desired_classes)}

            splits = SPLIT_RATIOS.keys()
            create_output_structure(current_output_dir, splits)
            create_class_csv(current_output_dir, splits, current_class_map)

            processing_plan = {
                "train": (train_patients, os.path.join(current_output_dir, 'train')),
                "valid": (valid_patients, os.path.join(current_output_dir, 'valid')),
                "test":  (test_patients, os.path.join(current_output_dir, 'test'))
            }

            for split_name, (patient_list, dest_dir) in processing_plan.items():
                print(f"\n--- Processing set '{split_name}' for '{config_name}' ---")
                process_and_convert_files(patient_list, ROOT_DIR, dest_dir, current_class_map, ENABLE_RESIZE, TARGET_SIZE)
                print(f"Set '{split_name}' finished.")

            print_summary_report(current_output_dir, total_images, splits)
            print(f"\n====== Dataset '{config_name}' generated successfully! ======\n")

            # --- FINAL STEP: mouth-detection crop ---
            if ENABLE_MOUTH_CROP and detector is not None:
                print(f"--- Starting mouth-detection crop for '{config_name}' ---")
                crop_dataset_in_place(current_output_dir, detector, splits)
                print("\n--- Crop finished. ---")
            else:
                print("--- Mouth-detection crop SKIPPED per configuration. ---")

        print("\nALL DATASETS GENERATED SUCCESSFULLY!")

In [ ]:
class SemanticSegmentationDataset(Dataset):
    """Image (semantic) segmentation dataset."""

    def __init__(self, root_dir, feature_extractor):
        """
        Args:
            root_dir (string): Root directory of the dataset containing the images + annotations.
            feature_extractor (SegFormerFeatureExtractor): feature extractor to prepare images + segmentation maps.
            train (bool): Whether to load "training" or "validation" images + annotations.
        """
        self.root_dir = root_dir
        self.feature_extractor = feature_extractor

        self.classes_csv_file = os.path.join(self.root_dir, "_classes.csv")
        with open(self.classes_csv_file, 'r') as fid:
            data = [l.split(',') for i,l in enumerate(fid) if i !=0]
        self.id2label = {x[0]:x[1] for x in data}

        image_file_names = [f for f in os.listdir(self.root_dir) if '.jpg' in f]
        mask_file_names = [f for f in os.listdir(self.root_dir) if '.png' in f]

        self.images = sorted(image_file_names)
        self.masks = sorted(mask_file_names)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = Image.open(os.path.join(self.root_dir, self.images[idx]))
        segmentation_map = Image.open(os.path.join(self.root_dir, self.masks[idx]))

        # randomly crop + pad both image and segmentation map to same size
        encoded_inputs = self.feature_extractor(image, segmentation_map, return_tensors="pt")

        for k,v in encoded_inputs.items():
          encoded_inputs[k].squeeze_() # remove batch dimension

        return encoded_inputs

In [ ]:
class SegformerFinetuner(pl.LightningModule):

    def __init__(self, id2label, train_dataloader=None, val_dataloader=None, test_dataloader=None, metrics_interval=100):
        super(SegformerFinetuner, self).__init__()
        self.id2label = id2label
        self.metrics_interval = metrics_interval
        self.train_dl = train_dataloader
        self.val_dl = val_dataloader
        self.test_dl = test_dataloader

        self.num_classes = len(id2label.keys())
        self.label2id = {v:k for k,v in self.id2label.items()}

        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",
            return_dict=False,
            num_labels=self.num_classes,
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True,
        )

        self.train_mean_iou = load_metric("mean_iou")
        self.val_mean_iou = load_metric("mean_iou")
        self.test_mean_iou = load_metric("mean_iou")

        # Lists to store outputs from each step
        self.validation_step_outputs = []
        # --- IMPORTANT FIX ---
        self.test_step_outputs = []  # List used to collect per-batch test outputs

    def forward(self, images, masks):
        outputs = self.model(pixel_values=images, labels=masks)
        return outputs

    def training_step(self, batch, batch_nb):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs[0], outputs[1]
        upsampled_logits = nn.functional.interpolate(
            logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.train_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(),
            references=masks.detach().cpu().numpy()
        )
        if batch_nb % self.metrics_interval == 0:
            metrics = self.train_mean_iou.compute(
                num_labels=self.num_classes, ignore_index=255, reduce_labels=False
            )
            metrics_log = {'loss': loss, "mean_iou": metrics["mean_iou"], "mean_accuracy": metrics["mean_accuracy"]}
            for k,v in metrics_log.items():
                self.log(k,v)
            return metrics_log
        else:
            return {'loss': loss}

    def validation_step(self, batch, batch_nb):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs[0], outputs[1]
        upsampled_logits = nn.functional.interpolate(
            logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.val_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(),
            references=masks.detach().cpu().numpy()
        )
        self.validation_step_outputs.append({'val_loss': loss})
        return {'val_loss': loss}

    def on_validation_epoch_end(self):
        metrics = self.val_mean_iou.compute(
              num_labels=self.num_classes, ignore_index=255, reduce_labels=False
        )
        avg_val_loss = torch.stack([x["val_loss"] for x in self.validation_step_outputs]).mean()
        val_mean_iou = metrics["mean_iou"]
        val_mean_accuracy = metrics["mean_accuracy"]
        metrics_log = {"val_loss": avg_val_loss, "val_mean_iou":val_mean_iou, "val_mean_accuracy":val_mean_accuracy}
        for k,v in metrics_log.items():
            self.log(k,v)
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_nb):
        images, masks = batch['pixel_values'], batch['labels']
        outputs = self(images, masks)
        loss, logits = outputs[0], outputs[1]
        upsampled_logits = nn.functional.interpolate(
            logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)
        self.test_mean_iou.add_batch(
            predictions=predicted.detach().cpu().numpy(),
            references=masks.detach().cpu().numpy()
        )
        # --- IMPORTANT FIX ---
        self.test_step_outputs.append({'test_loss': loss})  # Store the test loss
        return {'test_loss': loss}

    # --- IMPORTANT FIX ---
    def on_test_epoch_end(self):  # PyTorch Lightning >= 2.0 hook signature (no 'outputs' arg)
        metrics = self.test_mean_iou.compute(
              num_labels=self.num_classes, ignore_index=255, reduce_labels=False
        )

        # Use the list populated in test_step
        avg_test_loss = torch.stack([x["test_loss"] for x in self.test_step_outputs]).mean()
        test_mean_iou = metrics["mean_iou"]
        test_mean_accuracy = metrics["mean_accuracy"]

        metrics_log = {"test_loss": avg_test_loss, "test_mean_iou":test_mean_iou, "test_mean_accuracy":test_mean_accuracy}

        for k,v in metrics_log.items():
            self.log(k,v)

        # Clear the list for future runs
        self.test_step_outputs.clear()

    def configure_optimizers(self):
        return torch.optim.Adam([p for p in self.parameters() if p.requires_grad], lr=2e-05, eps=1e-08)

    def train_dataloader(self):
        return self.train_dl

    def val_dataloader(self):
        return self.val_dl

    def test_dataloader(self):
        return self.test_dl

In [ ]:
dataset_location = '/content/Datasets/ds_crown'

feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
feature_extractor.do_reduce_labels = False
feature_extractor.size = 128

train_dataset = SemanticSegmentationDataset(f"{dataset_location}/train/", feature_extractor)
val_dataset = SemanticSegmentationDataset(f"{dataset_location}/valid/", feature_extractor)
test_dataset = SemanticSegmentationDataset(f"{dataset_location}/test/", feature_extractor)

batch_size = 246
num_workers = 6
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, num_workers=num_workers)

segformer_finetuner = SegformerFinetuner(
    train_dataset.id2label,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=test_dataloader,
    metrics_interval=10,
)

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=50,
    verbose=False,
    mode="min",
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_mean_iou",      # Metric to monitor
    mode="max",                  # Maximize IoU (use "min" for losses such as val_loss)
    save_top_k=1,                # Save only the best model
    dirpath="checkpoints/",      # Directory where checkpoints are saved
    filename="best-model-{epoch}-{val_mean_iou:.4f}"  # Filename pattern for the saved model
)

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    callbacks=[early_stop_callback, checkpoint_callback],
    max_epochs=500,
    val_check_interval=len(train_dataloader),
)
trainer.fit(segformer_finetuner)

In [ ]:
!zip -r checkpoints.zip checkpoints/

In [ ]:
!zip -r lightning_logs.zip lightning_logs/

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/

In [ ]:
# NOTE: InReDD property - not publicly available, not distributed with this notebook.
caminho_do_checkpoint = "/content/checkpoints/best-model.ckpt"

print(f"Starting test with the specified checkpoint: {caminho_do_checkpoint}")

# Run the test with the EXPLICIT path
res = trainer.test(model=segformer_finetuner, ckpt_path=caminho_do_checkpoint)

print("\nTest results:", res)

In [ ]:
color_map = {
    0: (0, 0, 0),       # Class 0 (background) -> black
    1: (255, 0, 0),     # Class 1 (e.g. crown / ridge) -> red
}

# Visualization function
def prediction_to_vis(prediction):
    vis_shape = prediction.shape + (3,)
    vis = np.zeros(vis_shape)
    for i, c in color_map.items():
        vis[prediction == i] = color_map[i]
    return Image.fromarray(vis.astype(np.uint8))

# Helper to reverse image normalization for visualization
def unnormalize_image(tensor_image):
    # ImageNet means and standard deviations, as used by SegFormer
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    # Move the tensor to CPU, convert to numpy, and reorder axes from (C, H, W) to (H, W, C)
    image_np = tensor_image.cpu().numpy().transpose(1, 2, 0)

    # Undo normalization
    unnormalized_image = (image_np * std) + mean

    # Clamp values to [0, 1] for display
    unnormalized_image = np.clip(unnormalized_image, 0, 1)

    return unnormalized_image


# --- Visualization loop ---

# Take a single batch from the test dataloader
try:
    batch = next(iter(test_dataloader))
except StopIteration:
    print("Test dataloader is empty.")
    batch = None

if batch:
    # Put the model in evaluation mode
    segformer_finetuner.eval()
    segformer_finetuner.to('cuda' if torch.cuda.is_available() else 'cpu')

    # Move images to the correct device
    images = batch['pixel_values'].to(segformer_finetuner.device)
    masks = batch['labels']  # Keep mask on CPU for now

    # Run inference without computing the loss
    with torch.no_grad():
        # For inference, only the images are passed to the model
        outputs = segformer_finetuner.model(pixel_values=images, labels=None)

    logits = outputs[0]

    # Resize logits to the original mask size
    upsampled_logits = nn.functional.interpolate(
        logits,
        size=masks.shape[-2:],  # (H, W) of the original mask
        mode="bilinear",
        align_corners=False
    )

    # Get the predicted mask
    predicted_mask = upsampled_logits.argmax(dim=1).cpu().numpy()

    # Convert masks and original images to numpy for plotting
    masks_np = masks.cpu().numpy()
    images_np = images.cpu()  # Already on CPU for unnormalize()

    # --- Plotting ---
    # Set how many examples from the batch to plot
    n_plots = min(4, len(images))  # Plot up to 4 images or the batch size

    # Three columns: original image, ground-truth mask, predicted mask
    fig, axarr = plt.subplots(n_plots, 3, figsize=(10, n_plots * 3))

    # Adjust for the n_plots=1 case
    if n_plots == 1:
        axarr = np.array([axarr])

    for i in range(n_plots):
        # 1. Original image
        axarr[i, 0].imshow(unnormalize_image(images_np[i]))
        axarr[i, 0].set_title("Imagem Original")
        axarr[i, 0].axis('off')

        # 2. Ground-truth mask
        axarr[i, 1].imshow(prediction_to_vis(masks_np[i, :, :]))
        axarr[i, 1].set_title("Máscara Real (Ground Truth)")
        axarr[i, 1].axis('off')

        # 3. Predicted mask
        axarr[i, 2].imshow(prediction_to_vis(predicted_mask[i, :, :]))
        axarr[i, 2].set_title("Máscara Prevista pelo Modelo")
        axarr[i, 2].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
#Predict on a test image and overlay the mask on the original image
test_idx = 0
input_image_file = os.path.join(test_dataset.root_dir,test_dataset.images[test_idx])
input_image = Image.open(input_image_file)
test_batch = test_dataset[test_idx]
images, masks = test_batch['pixel_values'], test_batch['labels']
images = torch.unsqueeze(images, 0)
masks = torch.unsqueeze(masks, 0)
outputs = segformer_finetuner.model(images, masks)

loss, logits = outputs[0], outputs[1]

upsampled_logits = nn.functional.interpolate(
    logits,
    size=masks.shape[-2:],
    mode="bilinear",
    align_corners=False
)
predicted_mask = upsampled_logits.argmax(dim=1).cpu().numpy()
mask = prediction_to_vis(np.squeeze(masks))
mask = mask.resize(input_image.size)
mask = mask.convert("RGBA")
input_image = input_image.convert("RGBA")
overlay_img = Image.blend(input_image, mask, 0.5)

# YOLO 11 Training Section

In [ ]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# YOLOv11 datasets - local layout.
#
# NOTE: InReDD property - the YOLO-format datasets used for crown and
# alveolar-ridge segmentation are not publicly available. To re-run this
# notebook on equivalent data, place YOLOv11 datasets at:
#   data/yolo/crown/data.yaml      (with images/ and labels/ subfolders, Ultralytics spec)
#   data/yolo/alveolar/data.yaml
from ultralytics import YOLO

CROWN_DATA_YAML = 'data/yolo/crown/data.yaml'
ALVEOLAR_DATA_YAML = 'data/yolo/alveolar/data.yaml'


In [ ]:
!rm -r YOLOv11_Training/

In [ ]:
import torch
import gc

# Delete the model and any other large variables
try:
    del model
    # del other_large_variable
except NameError:
    print("No variables found to delete.")

# Force Python's garbage collector to free RAM
gc.collect()

# Clear PyTorch's GPU memory cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cache cleared.")

In [ ]:
import os

def remove_label_from_txt(file_path, label_prefix):
    """Removes lines starting with label_prefix from a text file."""
    try:
        with open(file_path, 'r') as f_in:
            lines = f_in.readlines()

        filtered_lines = [line for line in lines if not line.strip().startswith(label_prefix)]

        with open(file_path, 'w') as f_out:
            f_out.writelines(filtered_lines)

        print(f"Processed {file_path}: Removed lines starting with '{label_prefix}'")

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
    except Exception as e:
        print(f"An error occurred while processing {file_path}: {e}")

# Define the label prefix to remove
label_prefix_to_remove = '1' # Assuming '1' represents 'mouth'

# Define the dataset roots
yolo_dataset_roots = [
    "/content/crown-3/train/labels",
    "/content/crown-3/valid/labels",
    "/content/crown-3/test/labels",
    "/content/alveolar-2/train/labels",
    "/content/alveolar-2/valid/labels",
    "/content/alveolar-2/test/labels",
]

# Iterate through the label directories and process each .txt file
for root_dir in yolo_dataset_roots:
    if os.path.exists(root_dir) and os.path.isdir(root_dir):
        print(f"\nProcessing directory: {root_dir}")
        for filename in os.listdir(root_dir):
            if filename.endswith(".txt"):
                file_path = os.path.join(root_dir, filename)
                remove_label_from_txt(file_path, label_prefix_to_remove)
    else:
        print(f"\nWarning: Directory not found or not a directory: {root_dir}")

print("\nFinished processing all specified label directories.")


## Training Alveolar Ridge

In [ ]:
from time import sleep
for i in range(40):
  sleep(120)

In [ ]:
model = YOLO('yolo11x-seg.pt')
model.train(model='yolo11x-seg.pt', data=ALVEOLAR_DATA_YAML, epochs=1000, patience=100, batch=8, imgsz=1280, device=0, project='YOLOv11_Training', name='run_RTX5070_alveolar_2')  # train the model

In [ ]:
import os
import time
import shutil
import subprocess

# --- Configuration ---
# The final model file that indicates training is complete.
final_model_path = '/content/YOLOv11_Training/run_RTX5070_alveolar_2/weights/best.pt'

# The folder you want to zip.
folder_to_zip = '/content/YOLOv11_Training'

# Name for the output zip file.
zip_output_filename = 'YOLOv11_Training_results.zip'
# ---------------------

print("Setup complete. Waiting for training to finish...")
print(f"Will look for file: {final_model_path}")

# Loop that checks for the file every 10 minutes
while not os.path.exists(final_model_path):
    print(f"Still waiting... Checking again in 10 minutes. ({time.ctime()})")
    time.sleep(600) # Wait for 600 seconds (10 minutes)

print("\nTraining finished! Found the model file.")

try:
    # 1. Zip the entire training folder
    print(f"\nZipping folder: {folder_to_zip}...")
    # shutil.make_archive returns the path of the created archive, but we already know it
    shutil.make_archive(zip_output_filename.replace('.zip', ''), 'zip', folder_to_zip)
    print(f"Successfully created {zip_output_filename}")

    # 2. Upload the zip file using curl to transfer.sh
    print(f"\nUploading {zip_output_filename} to transfer.sh...")
    # The command needs to be formatted correctly for subprocess
    command = f"curl --upload-file ./{zip_output_filename} https://transfer.sh/{zip_output_filename}"

    # Run the command and capture the output
    process = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # The download link is the standard output
    download_link = process.stdout.decode('utf-8').strip()

    print("\n\nAll tasks complete!")
    print("--------------------------------------------------")
    print("YOUR DOWNLOAD LINK (SAVE THIS!):")
    print(download_link)
    print("--------------------------------------------------")
    print("The link is valid for 14 days. You can now safely close your browser.")

except Exception as e:
    print(f"\nAn error occurred during the process: {e}")
    # If there was an error, print the stderr from the subprocess as well
    if 'process' in locals():
        print("Error details from upload command:")
        print(process.stderr.decode('utf-8'))

## Training Crown

In [ ]:
model = YOLO('yolo11x-seg.pt')
model.train(model='yolo11x-seg.pt', data=CROWN_DATA_YAML, epochs=1000, patience=100, batch=8, imgsz=1280, device=0, project='YOLOv11_Training', name='run_RTX5070')  # train the model

In [ ]:
!zip -r YOLOv11_Training.zip YOLOv11_Training/